In [27]:
import pint
import json
import jsonata
from IPython.display import JSON, HTML
import pandas as pd

# ── ANSI palette ──────────────────────────────────────────
RESET, BOLD, DIM = "\033[0m", "\033[1m", "\033[2m"
CYAN, GREEN, YELLOW = "\033[36m", "\033[32m", "\033[33m"

reg = pint.UnitRegistry()
reg.define('kg = kilogram')
reg.define('mg = milligram')
reg.define('mcg = microgram')
reg.define('mL = milliliter')
reg.define('ug = microgram')
reg.define('ng = nanogram')
reg.define('cg = centigram')
reg.define('dL = deciliter')
reg.define('L = liter')
reg.define('g = gram')
reg.define('gm = gram')
reg.define('equivalent = [substance_charge] = Eq')
reg.define('meq = 0.001 equivalent = mEq')
reg.define('mEq = 0.001 equivalent = meq')
# Activity / arbitrary units family — all dimensionally [activity]
reg.define('unit = [activity] = U = IU = iu = USP_U = arb_U')
reg.define('milliunit = 0.001 unit = mU = mIU')
reg.define('million_unit = 1e6 unit = MU = MIU')
mass = 1000 * reg.mg
vol = 1000 * reg.mL
conc = mass / vol

UNITS = ("mg/mL", "cg/dL", "ug/mL", "ng/mL", "g/mL", "mg/L", "ug/L", "g/L")


def show_conc(conc, units=UNITS, prec=6):
      """Print a concentration in multiple units as an aligned, colorized table."""
      print(f"{BOLD}{CYAN}CONC:{RESET} {BOLD}{conc:~P}{RESET}")
      w = max(map(len, units))
      for u in units:
            q = conc.to(u)
            print(f"-> {GREEN}{q.magnitude:>7,.{prec}g}{RESET}"
                  f" {RESET} {YELLOW}{u:<{w}}{RESET}")


show_conc(conc)

CONC: 1.0 mg/mL
->       1  mg/mL
->      10  cg/dL
->   1,000  ug/mL
->   1e+06  ng/mL
->   0.001  g/mL 
->   1,000  mg/L 
->   1e+06  ug/L 
->       1  g/L  


## FDA

## Functions

In [28]:
import re
import pint
from fda import clean_labeler_name

_STRENGTH_RE = re.compile(
      r'^\s*'
      r'(?P<num_val>\d+(?:\.\d+)?)\s*(?P<num_unit>[a-zA-Zµμ%]+)'
      r'\s*(?:/\s*'
      r'(?P<den_val>\d+(?:\.\d+)?)?\s*(?P<den_unit>[a-zA-Zµμ]+)'
      r')?\s*$'
)
_UCUM_BRACKET_RE = re.compile(r"\[([^\]]+)\]")
# language=JSONata
_JSONATA_PTYPE_EXTRACTOR = r"$ ~> |**[description]|{'product_type': $reverse($match(description, /\b[A-Z\s,\-]{3,}/).match)[0]}|"


def extract_product_type(obj):
      return jsonata.Jsonata(_JSONATA_PTYPE_EXTRACTOR).evaluate(obj)


def _normalize_ucum(s: str) -> str:
      # [USP'U] -> USP_U, [iU] -> iU, [arb'U] -> arb_U
      return _UCUM_BRACKET_RE.sub(
            lambda m: m.group(1).replace("'", "_").replace(".", "_"),
            s,
      )


def parse_strength(strength_str: str) -> pint.Quantity[pint.Unit]:
      """
      Parse a pharmaceutical strength string into a pint Quantity.
      """
      if not isinstance(strength_str, str) or not strength_str.strip():
            raise ValueError(f"Empty or non-string strength: {strength_str!r}")

      s_str = strength_str.strip().replace('μ', 'u').replace('µ', 'u')  # normalize micro
      s_norm = _normalize_ucum(s_str)

      m = _STRENGTH_RE.match(s_norm)
      if not m:
            # Fall back to letting pint try directly (handles odd cases pint knows about)
            try:
                  return reg.parse_expression(s_norm)
            except Exception as e:
                  raise ValueError(f"Could not parse strength {strength_str!r}: {e}") from e

      num_val = float(m.group('num_val'))
      num_unit = m.group('num_unit')
      den_unit = m.group('den_unit')
      den_val = float(m.group('den_val')) if m.group('den_val') else 1.0

      try:
            numerator = num_val * reg.parse_expression(num_unit)
      except pint.errors.UndefinedUnitError as e:
            raise ValueError(f"Unknown numerator unit in {strength_str!r}: {num_unit}") from e

      if den_unit is None:
            return numerator

      try:
            denominator = den_val * reg.parse_expression(den_unit)
      except pint.errors.UndefinedUnitError as e:
            raise ValueError(f"Unknown denominator unit in {strength_str!r}: {den_unit}") from e

      if denominator.magnitude == 0:
            raise ValueError(f"Zero denominator in {strength_str!r}")

      return numerator / denominator


def parse_strength_in_context(obj):
      if isinstance(obj, dict):
            return {
                  k: (parse_strength(v) if k == 'strength' and isinstance(v, str)
                      else parse_strength_in_context(v))
                  for k, v in obj.items()
            }
      if isinstance(obj, list):
            return [parse_strength_in_context(x) for x in obj]
      return obj


def get_product_type(packaging):
      try:
            return packaging[0]['product_type']
      except (IndexError, KeyError, TypeError):
            return None


## Usage

In [29]:
from rxocrpl.fda import lookup_generic_name, lookup_ndc_package

kpo4 = lookup_generic_name('potassium phosphates', dosage_form='INJECTION')
kpo4 = extract_product_type(kpo4)
JSON(kpo4)

<IPython.core.display.JSON object>

In [30]:
# language=JSONata
data = jsonata.Jsonata("""
    *.packaging[$contains(description, 'BAG')].{
        'labeler_name': %.labeler_name,
        'brand_name': %.brand_name,
        'generic_name': %.generic_name,
        'product_ndc': %.product_ndc,
        'active_ingredients': %.active_ingredients,
        'package_ndc': package_ndc,
        'description': description,
        'product_type': product_type
    }
""").evaluate(kpo4)
df = pd.DataFrame(data)
df['labeler_name'] = df['labeler_name'].apply(clean_labeler_name)
df

,labeler_name,brand_name,generic_name,product_ndc,active_ingredients,package_ndc,description,product_type
0,Fresenius Kabi,Potassium Phosphates,Potassium Phosphates,65219-658,"[{'name': 'DIBASIC POTASSIUM PHOSPHATE', 'stre...",65219-658-25,30 POUCH in 1 CARTON (65219-658-25) / 1 BAG i...,BAG
1,Fresenius Kabi,Potassium Phosphates,Potassium Phosphates,65219-656,"[{'name': 'DIBASIC POTASSIUM PHOSPHATE', 'stre...",65219-656-10,24 POUCH in 1 CARTON (65219-656-10) / 1 BAG i...,BAG
2,Amneal,POTASSIUM PHOSPHATES,Potassium Phosphates in sodium chloride,70121-1722,"[{'name': 'POTASSIUM PHOSPHATE, DIBASIC', 'str...",70121-1722-9,24 POUCH in 1 CARTON (70121-1722-9) / 1 BAG i...,BAG


In [31]:
# language=JSONata
print('BAGS:', jsonata.Jsonata("packaging[$contains(description, 'BAG')] ~> $count()").evaluate(kpo4))
# language=JSONata
print('VIALS:', jsonata.Jsonata("packaging[$contains(description, 'VIAL')] ~> $count()").evaluate(kpo4))

BAGS: 3
VIALS: 11


In [32]:
res = jsonata.Jsonata("$.[labeler_name, [active_ingredients]]").evaluate(kpo4)

rows = []

for labeler_name, ingredients in res:
      for ing in ingredients:
            s = parse_strength(ing["strength"])  # pint.Quantity

            strength_mg_ml = s.to("mg/mL")

            rows.append({
                  "labeler_name": labeler_name,
                  "name": ing["name"],
                  "strength": str(s),
                  "str/mL": float(strength_mg_ml.magnitude),
                  "str_unit": str((strength_mg_ml * reg.mL).units),
            })

df = pd.DataFrame(rows)
df

,labeler_name,name,strength,str/mL,str_unit
0,Caplin Steriles Limited,"POTASSIUM PHOSPHATE, DIBASIC",236.0 mg / mL,236.00,mg
1,Caplin Steriles Limited,"POTASSIUM PHOSPHATE, MONOBASIC",224.0 mg / mL,224.00,mg
2,Caplin Steriles Limited,"POTASSIUM PHOSPHATE, DIBASIC",236.0 mg / mL,236.00,mg
3,Caplin Steriles Limited,"POTASSIUM PHOSPHATE, MONOBASIC",224.0 mg / mL,224.00,mg
4,Caplin Steriles Limited,"POTASSIUM PHOSPHATE, DIBASIC",236.0 mg / mL,236.00,mg
5,Caplin Steriles Limited,"POTASSIUM PHOSPHATE, MONOBASIC",224.0 mg / mL,224.00,mg
6,"American Regent, Inc.","POTASSIUM PHOSPHATE, DIBASIC",236.0 mg / mL,236.00,mg
7,"American Regent, Inc.","POTASSIUM PHOSPHATE, MONOBASIC",224.0 mg / mL,224.00,mg
8,"GLENMARK PHARMACEUTICALS INC., USA",DIBASIC POTASSIUM PHOSPHATE,236.0 mg / mL,236.00,mg
9,"GLENMARK PHARMACEUTICALS INC., USA",MONOBASIC POTASSIUM PHOSPHATE,224.0 mg / mL,224.00,mg


In [33]:
ns_bag = lookup_generic_name(ingredient_names=['sodium chloride'], dosage_form='INJECTION', max_active_ingredients=1)
ns_bag = extract_product_type(ns_bag)
df = pd.DataFrame(ns_bag)
df['product_type'] = df['packaging'].apply(get_product_type)
df

,labeler_name,brand_name,generic_name,product_ndc,active_ingredients,dosage_form,route,packaging,rxcui,_matched_via,product_type
0,"Fresenius Kabi USA, LLC",Sodium chloride,SODIUM CHLORIDE,65219-504,"[{'name': 'SODIUM CHLORIDE', 'strength': '.9 g...","INJECTION, SOLUTION",[IRRIGATION],"[{'package_ndc': '65219-504-20', 'description'...",None,other,BAG
1,"Fresenius Kabi USA, LLC",Sodium chloride,SODIUM CHLORIDE,65219-506,"[{'name': 'SODIUM CHLORIDE', 'strength': '.9 g...","INJECTION, SOLUTION",[IRRIGATION],"[{'package_ndc': '65219-506-30', 'description'...",None,other,BAG
2,"Fresenius Kabi USA, LLC",Sodium Chloride,Sodium Chloride,63323-088,"[{'name': 'SODIUM CHLORIDE', 'strength': '234 ...","INJECTION, SOLUTION",[INTRAVENOUS],"[{'package_ndc': '63323-088-61', 'description'...",None,other,"VIAL, PHARMACY BULK PACKAGE"
3,Hikma Pharmaceuticals USA Inc.,Sodium Chloride,Sodium Chloride,0641-0497,"[{'name': 'SODIUM CHLORIDE', 'strength': '9 mg...",INJECTION,"[INTRAMUSCULAR, INTRAVENOUS, SUBCUTANEOUS]","[{'package_ndc': '0641-0497-25', 'description'...",None,other,VIAL
4,"Fresenius Kabi USA, LLC",Sodium Chloride,Sodium Chloride,63323-095,"[{'name': 'SODIUM CHLORIDE', 'strength': '4 me...","INJECTION, SOLUTION, CONCENTRATE",[INTRAVENOUS],"[{'package_ndc': '63323-095-61', 'description'...",None,other,"VIAL, PLASTIC"
5,"Fresenius Kabi USA, LLC",Sodium Chloride,Sodium Chloride,63323-090,"[{'name': 'SODIUM CHLORIDE', 'strength': '2.5 ...","INJECTION, SOLUTION, CONCENTRATE",[INTRAVENOUS],"[{'package_ndc': '63323-090-20', 'description'...",None,other,"VIAL, PLASTIC"
6,"Fresenius Kabi USA, LLC",Sodium Chloride,Sodium Chloride,63323-093,"[{'name': 'SODIUM CHLORIDE', 'strength': '4 me...","INJECTION, SOLUTION, CONCENTRATE",[INTRAVENOUS],"[{'package_ndc': '63323-093-30', 'description'...",None,other,"VIAL, PLASTIC"
7,"Fresenius Kabi USA, LLC",Sodium Chloride,Sodium Chloride,63323-099,"[{'name': 'SODIUM CHLORIDE', 'strength': '4 me...","INJECTION, SOLUTION, CONCENTRATE",[INTRAVENOUS],"[{'package_ndc': '63323-099-63', 'description'...",None,other,"VIAL, PLASTIC"
8,Baxter Healthcare Corporation,Sodium Chloride,Sodium Chloride,0338-0196,"[{'name': 'SODIUM CHLORIDE', 'strength': '900 ...",INJECTION,[INTRAVENOUS],"[{'package_ndc': '0338-0196-04', 'description'...",None,other,BAG
9,"Fresenius Medical Care Renal Therapies Group, LLC",Sodium Chloride,Sodium Chloride,49230-300,"[{'name': 'SODIUM CHLORIDE', 'strength': '900 ...",INJECTION,[INTRAVENOUS],"[{'package_ndc': '49230-300-10', 'description'...",None,other,BAG


In [34]:
avicaz = lookup_generic_name(brand_name='Avycaz', labelers_cleanup=True)
avicaz = extract_product_type(avicaz)
df = pd.DataFrame(avicaz)
df


,labeler_name,brand_name,generic_name,product_ndc,active_ingredients,dosage_form,route,packaging,rxcui,_matched_via
0,Allergan,AVYCAZ,"ceftazidime, avibactam",0456-2700,"[{'name': 'AVIBACTAM SODIUM', 'strength': '.5 ...","POWDER, FOR SOLUTION",[INTRAVENOUS],"[{'package_ndc': '0456-2700-10', 'description'...",None,brand_name_narrowed


In [35]:
ceft_avi = lookup_generic_name("ceftazidime, avibactam", labelers_cleanup=True)

In [36]:
df = pd.DataFrame(jsonata.Jsonata("**[product_ndc]").evaluate(ns_bag))

df['labeler_name'] = df['labeler_name'].apply(clean_labeler_name)

df['product_type'] = df['packaging'].apply(get_product_type)
df['ingr_count'] = df['active_ingredients'].apply(len)

bag_mask = df['product_type'].str.contains('BAG', na=False)
ingr_mask = df['ingr_count'] == 1

mask = bag_mask & ingr_mask
df = df[mask]
df

,labeler_name,brand_name,generic_name,product_ndc,active_ingredients,dosage_form,route,packaging,rxcui,_matched_via,product_type,ingr_count
0,Fresenius Kabi,Sodium chloride,SODIUM CHLORIDE,65219-504,"[{'name': 'SODIUM CHLORIDE', 'strength': '.9 g...","INJECTION, SOLUTION",[IRRIGATION],"[{'package_ndc': '65219-504-20', 'description'...",None,other,BAG,1
1,Fresenius Kabi,Sodium chloride,SODIUM CHLORIDE,65219-506,"[{'name': 'SODIUM CHLORIDE', 'strength': '.9 g...","INJECTION, SOLUTION",[IRRIGATION],"[{'package_ndc': '65219-506-30', 'description'...",None,other,BAG,1
8,Baxter,Sodium Chloride,Sodium Chloride,0338-0196,"[{'name': 'SODIUM CHLORIDE', 'strength': '900 ...",INJECTION,[INTRAVENOUS],"[{'package_ndc': '0338-0196-04', 'description'...",None,other,BAG,1
9,Fresenius Medical Care Renal Therapies,Sodium Chloride,Sodium Chloride,49230-300,"[{'name': 'SODIUM CHLORIDE', 'strength': '900 ...",INJECTION,[INTRAVENOUS],"[{'package_ndc': '49230-300-10', 'description'...",None,other,BAG,1
10,Baxter,SODIUM CHLORIDE,sodium chloride,0338-9657,"[{'name': 'SODIUM CHLORIDE', 'strength': '9 g/...","INJECTION, SOLUTION",[INTRAVENOUS],"[{'package_ndc': '0338-9657-75', 'description'...",None,other,BAG,1
11,SOLA,Sodium Chloride,Sodium Chloride,70512-841,"[{'name': 'SODIUM CHLORIDE', 'strength': '9 g/...","INJECTION, SOLUTION",[INTRAVENOUS],"[{'package_ndc': '70512-841-06', 'description'...",None,other,BAG,1
12,ProPharma Distribution,Sodium Chloride,Sodium Chloride,84549-049,"[{'name': 'SODIUM CHLORIDE', 'strength': '9 g/...","INJECTION, SOLUTION",[INTRAVENOUS],"[{'package_ndc': '84549-049-48', 'description'...",None,other,BAG,1
13,Baxter,SODIUM CHLORIDE,sodium chloride,0338-9659,"[{'name': 'SODIUM CHLORIDE', 'strength': '9 g/...","INJECTION, SOLUTION",[INTRAVENOUS],"[{'package_ndc': '0338-9659-75', 'description'...",None,other,BAG,1
19,Baxter,SODIUM CHLORIDE,sodium chloride,0338-9661,"[{'name': 'SODIUM CHLORIDE', 'strength': '9 g/...","INJECTION, SOLUTION",[INTRAVENOUS],"[{'package_ndc': '0338-9661-60', 'description'...",None,other,BAG,1
20,Fresenius Kabi,Sodium Chloride,Sodium Chloride,63323-530,"[{'name': 'SODIUM CHLORIDE', 'strength': '3 g/...","INJECTION, SOLUTION",[INTRAVENOUS],"[{'package_ndc': '63323-530-75', 'description'...",None,other,BAG,1


In [37]:
vaso = lookup_generic_name('vasopressin', dosage_form='INJECTION')
data = parse_strength_in_context(vaso)
df = pd.DataFrame(data)
df

,labeler_name,brand_name,generic_name,product_ndc,active_ingredients,dosage_form,route,packaging,rxcui,_matched_via
0,"Medical Purchasing Solutions, LLC",Vasostrict,Vasopressin,71872-7264,"[{'name': 'VASOPRESSIN, UNSPECIFIED', 'strengt...",INJECTION,[INTRAVENOUS],"[{'package_ndc': '71872-7264-1', 'description'...",None,generic_name
1,"HF Acquisition Co LLC, DBA HealthFirst",VASOPRESSIN,VASOPRESSIN,51662-1623,"[{'name': 'VASOPRESSIN, UNSPECIFIED', 'strengt...",INJECTION,[INTRAVENOUS],"[{'package_ndc': '51662-1623-1', 'description'...",None,generic_name
2,"Dr. Reddy's Laboratories, Inc.",vasopressin,Vasopressin,43598-914,"[{'name': 'VASOPRESSIN, UNSPECIFIED', 'strengt...",INJECTION,[INTRAVENOUS],"[{'package_ndc': '43598-914-06', 'description'...",None,generic_name
3,"HF Acquisition Co LLC, DBA HealthFirst",VASOPRESSIN,VASOPRESSIN,51662-1314,"[{'name': 'VASOPRESSIN, UNSPECIFIED', 'strengt...",INJECTION,[INTRAVENOUS],"[{'package_ndc': '51662-1314-1', 'description'...",None,generic_name
4,ProPharma Distribution,VASOPRESSIN,VASOPRESSIN,84549-370,"[{'name': 'VASOPRESSIN', 'strength': 20.0 unit...",INJECTION,[INTRAVENOUS],"[{'package_ndc': '84549-370-25', 'description'...",None,generic_name
5,"Medical Purchasing Solutions, LLC",VASOPRESSIN,VASOPRESSIN,71872-7306,"[{'name': 'VASOPRESSIN', 'strength': 20.0 unit...",INJECTION,[INTRAVENOUS],"[{'package_ndc': '71872-7306-1', 'description'...",None,generic_name
6,"Amphastar Pharmaceuticals, Inc.",Vasopressin,Vasopressin,0548-9701,"[{'name': 'VASOPRESSIN', 'strength': 20.0 unit...",INJECTION,[INTRAVENOUS],"[{'package_ndc': '0548-9701-00', 'description'...",None,generic_name
7,"Civica, Inc.",Vasopressin,Vasopressin,72572-860,"[{'name': 'VASOPRESSIN', 'strength': 20.0 unit...","INJECTION, SOLUTION",[INTRAVENOUS],"[{'package_ndc': '72572-860-25', 'description'...",None,generic_name
8,Sagent Pharmaceuticals,Vasopressin,Vasopressin,25021-474,"[{'name': 'VASOPRESSIN', 'strength': 20.0 unit...","INJECTION, SOLUTION",[INTRAVENOUS],"[{'package_ndc': '25021-474-01', 'description'...",None,generic_name
9,Amneal Pharmaceuticals LLC,vasopressin,vasopressin,70121-1642,"[{'name': 'VASOPRESSIN', 'strength': 20.0 unit...",INJECTION,[INTRAVENOUS],"[{'package_ndc': '70121-1642-2', 'description'...",None,generic_name


In [38]:
unasyn = lookup_generic_name("ampicillin")
unasyn = extract_product_type(unasyn)
df = pd.DataFrame(unasyn)
df

,labeler_name,brand_name,generic_name,product_ndc,active_ingredients,dosage_form,route,packaging,rxcui,_matched_via
0,"Medical Purchasing Solutions, LLC",Ampicillin,Ampicillin,71872-7249,"[{'name': 'AMPICILLIN SODIUM', 'strength': '1 ...","INJECTION, POWDER, FOR SOLUTION","[INTRAMUSCULAR, INTRAVENOUS]","[{'package_ndc': '71872-7249-1', 'description'...",None,generic_name
1,"Civica, Inc.",Ampicillin sodium,Ampicillin,72572-093,"[{'name': 'AMPICILLIN SODIUM', 'strength': '1 ...","INJECTION, POWDER, FOR SOLUTION","[INTRAMUSCULAR, INTRAVENOUS]","[{'package_ndc': '72572-093-10', 'description'...",None,generic_name
2,REMEDYREPACK INC.,Ampicillin,Ampicillin,70518-2607,"[{'name': 'AMPICILLIN TRIHYDRATE', 'strength':...",CAPSULE,[ORAL],"[{'package_ndc': '70518-2607-1', 'description'...",None,generic_name
3,"HF Acquisition Co LLC, DBA HealthFirst",AMPICILLIN,AMPICILLIN,51662-1560,"[{'name': 'AMPICILLIN SODIUM', 'strength': '2 ...","INJECTION, POWDER, FOR SOLUTION",[INTRAVENOUS],"[{'package_ndc': '51662-1560-3', 'description'...",None,generic_name
4,"Medical Purchasing Solutions, LLC",Ampicillin,Ampicillin,71872-7240,"[{'name': 'AMPICILLIN SODIUM', 'strength': '2 ...","INJECTION, POWDER, FOR SOLUTION","[INTRAMUSCULAR, INTRAVENOUS]","[{'package_ndc': '71872-7240-1', 'description'...",None,generic_name
...,...,...,...,...,...,...,...,...,...,...
95,Sandoz Inc,Ampicillin,Ampicillin sodium,0781-9404,"[{'name': 'AMPICILLIN SODIUM', 'strength': '1 ...","INJECTION, POWDER, FOR SOLUTION","[INTRAMUSCULAR, INTRAVENOUS]","[{'package_ndc': '0781-9404-95', 'description'...",None,generic_name
96,"Avenacy, LLC",Ampicillin and Sulbactam,ampicillin and sulbactam,83634-108,"[{'name': 'AMPICILLIN SODIUM', 'strength': '10...","INJECTION, POWDER, FOR SOLUTION",[INTRAVENOUS],"[{'package_ndc': '83634-108-99', 'description'...",None,generic_name
97,ONESOURCE SPECIALTY PHARMA LIMITED,Ampicillin Sodium and Sulbactam Sodium,"ampicillin sodium, sulbactam sodium",83270-308,"[{'name': 'AMPICILLIN SODIUM', 'strength': '10...","INJECTION, POWDER, FOR SOLUTION","[INTRAMUSCULAR, INTRAVENOUS]","[{'package_ndc': '83270-308-01', 'description'...",None,generic_name
98,"Avenacy, LLC",Ampicillin and Sulbactam,ampicillin and sulbactam,83634-109,"[{'name': 'AMPICILLIN SODIUM', 'strength': '1 ...","INJECTION, POWDER, FOR SOLUTION","[INTRAMUSCULAR, INTRAVENOUS]","[{'package_ndc': '83634-109-20', 'description'...",None,generic_name


In [39]:
result = jsonata.Jsonata("$.[active_ingredients.[name, strength]]").evaluate(vaso)

In [40]:
dxs = jsonata.Jsonata("**.description").evaluate(vaso)
for dx in dxs:
      print(dx)
      print(re.findall(r'\b[A-Z|\s|,|\-]{3,}', dx)[-1])

1 VIAL in 1 BAG (71872-7264-1)  / 1 mL in 1 VIAL
 VIAL
1 mL in 1 VIAL, MULTI-DOSE (51662-1623-1)
 VIAL, MULTI-DOSE 
1 VIAL, MULTI-DOSE in 1 POUCH (51662-1623-2)  / 1 mL in 1 VIAL, MULTI-DOSE
 VIAL, MULTI-DOSE
25 VIAL in 1 CARTON (43598-914-06)  / 1 mL in 1 VIAL (43598-914-11)
 VIAL 
25 VIAL in 1 CARTON (43598-914-25)  / 1 mL in 1 VIAL (43598-914-11)
 VIAL 
1 mL in 1 VIAL (51662-1314-1)
 VIAL 
1 VIAL in 1 POUCH (51662-1314-2)  / 1 mL in 1 VIAL
 VIAL
1 mL in 1 VIAL, MULTI-DOSE (84549-370-25)
 VIAL, MULTI-DOSE 
1 VIAL, MULTI-DOSE in 1 BAG (71872-7306-1)  / 1 mL in 1 VIAL, MULTI-DOSE
 VIAL, MULTI-DOSE
1 VIAL in 1 CARTON (0548-9701-00)  / 1 mL in 1 VIAL
 VIAL
25 VIAL, SINGLE-DOSE in 1 CARTON (72572-860-25)  / 1 mL in 1 VIAL, SINGLE-DOSE
 VIAL, SINGLE-DOSE
25 VIAL in 1 CARTON (25021-474-01)  / 1 mL in 1 VIAL
 VIAL
1 VIAL in 1 CARTON (70121-1642-2)  / 1 mL in 1 VIAL (70121-1642-1)
 VIAL 
25 VIAL in 1 CARTON (70121-1642-5)  / 1 mL in 1 VIAL (70121-1642-1)
 VIAL 
10 VIAL in 1 CARTON (70121-1642

In [41]:
# language=JSONata
res = jsonata.Jsonata("""$.{labeler_name:
                        [active_ingredients.{
                              'name': name,
                              'strength': strength,
                              'product_type': product_type,
                              'product_ndc': %.product_ndc
                        },
                        [**.description]]
                  }""").evaluate(vaso)
# data = parse_strength_in_context(res)
JSON(res)

<IPython.core.display.JSON object>

In [45]:
import pandas as pd
import re
import json

records = []
for entry in res:
      for labeler, contents in entry.items():
            drug_info = contents[0]  # {"name": ..., "strength": ..., "product_ndc": ...}
            packages = contents[1]  # ["1 mL in 1 VIAL (NDC-1)", ...]

            for pkg_str in packages:
                  # Extract all NDCs from the package string (there can be 2 — carton + unit)
                  ndcs = re.findall(r'\(([^)]+)\)', pkg_str)

                  records.append({
                        "labeler": labeler,
                        "name": drug_info["name"],
                        "strength": drug_info["strength"],
                        "product_ndc": drug_info["product_ndc"],
                        "package_desc": pkg_str,
                        "package_ndc": ndcs[0] if ndcs else None,  # outermost (carton/bag)
                        "unit_ndc": ndcs[1] if len(ndcs) > 1 else None,  # inner (vial)
                  })

df = pd.DataFrame(records)
df.groupby('product_ndc').agg({
      'labeler': 'first',
      'name': 'first',
      'strength': 'first',
      'package_desc': lambda x: list(x),
      'package_ndc': lambda x: list(x),
      'unit_ndc': lambda x: list(x),
}).reset_index()

,product_ndc,labeler,name,strength,package_desc,package_ndc,unit_ndc
0,0338-9640,Baxter Healthcare Corporation,VASOPRESSIN,20 [USP'U]/100mL,[12 BAG in 1 CARTON (0338-9640-12) / 100 mL i...,[0338-9640-12],[nan]
1,0338-9647,Baxter Healthcare Corporation,VASOPRESSIN,40 [USP'U]/100mL,[12 BAG in 1 CARTON (0338-9647-12) / 100 mL i...,[0338-9647-12],[nan]
2,0517-1020,"American Regent, Inc.",VASOPRESSIN,20 [USP'U]/mL,"[25 VIAL, SINGLE-DOSE in 1 CARTON (0517-1020-2...",[0517-1020-25],[0517-1020-01]
3,0517-1030,"American Regent, Inc.",VASOPRESSIN,20 [USP'U]/mL,"[1 VIAL, MULTI-DOSE in 1 CARTON (0517-1030-01)...",[0517-1030-01],[nan]
4,0548-9701,"Amphastar Pharmaceuticals, Inc.",VASOPRESSIN,20 [USP'U]/mL,[1 VIAL in 1 CARTON (0548-9701-00) / 1 mL in ...,[0548-9701-00],[nan]
5,25021-474,Sagent Pharmaceuticals,VASOPRESSIN,20 [USP'U]/mL,[25 VIAL in 1 CARTON (25021-474-01) / 1 mL in...,[25021-474-01],[nan]
6,42023-164,"Par Health USA, LLC","VASOPRESSIN, UNSPECIFIED",20 [USP'U]/mL,[10 VIAL in 1 CARTON (42023-164-10) / 1 mL in...,"[42023-164-10, 42023-164-25, 42023-164-83]","[42023-164-01, 42023-164-01, nan]"
7,42023-190,"Par Health USA, LLC","VASOPRESSIN, UNSPECIFIED",20 [USP'U]/mL,[1 VIAL in 1 CARTON (42023-190-01) / 10 mL in...,[42023-190-01],[nan]
8,42023-219,"Par Health USA, LLC","VASOPRESSIN, UNSPECIFIED",.4 [USP'U]/mL,[10 VIAL in 1 CARTON (42023-219-10) / 100 mL ...,[42023-219-10],[42023-219-01]
9,42023-237,"Par Health USA, LLC","VASOPRESSIN, UNSPECIFIED",.2 [USP'U]/mL,[10 VIAL in 1 CARTON (42023-237-10) / 100 mL ...,[42023-237-10],[42023-237-01]


# Filterable Drugs

In [ ]:
def get_inline_filter_drugs():
      url = r"http://pmc.ncbi.nlm.nih.gov/articles/PMC11907493/table/table1-00185787251324867"
      df = pd.read_html(url)[0]
      # remove NBSP chars
      df['Drug'] = df['Drug'].str.replace(r'\xa0', ' ')
      df['brand'] = df['Drug'].str.findall(r'\((.*?)\)')
      return df


filterable_drugs = get_inline_filter_drugs()
filterable_drugs